# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 29 · Round 13: three-arm feature ablation

`mask`: availability indicators with 12 values zero. `core`: six audited values. `full`: all twelve values. Identical capacity, initialization, optimizer exposure, masks, rows and labels. No ensemble or adaptive stacking with the other round.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT=Path('/home/sagemaker-user/nfl_feature_rounds12_13')
OUT=Path('/home/sagemaker-user/nfl-feature-round13-results')
PY=Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
ROUND=13
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract the combined package first.')
sys.path.insert(0,str(KIT))
import visuals
pio.renderers.default='plotly_mimetype'
def run(stage,*options):
    p=subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage,'--round',str(ROUND),*options],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end='')
        code=p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        try:p.wait(timeout=10)
        except subprocess.TimeoutExpired:p.kill();p.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve outputs, export the report, and do not change settings.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()

## Cached CPU runtime
Require `family_runtime_ready`. Executes 28 synthetic model/metric tests using the existing pinned CPU environment. Offline only. Stop on a cache or version mismatch.

In [ ]:
run('runtime')

## Training-only throughput gate
Require `family_profile_passed`. Twenty-four disposable optimizer steps are engineering checks, not scientific fits. A `family_profile_budget_stop` means stop and export; do not raise the budget.

In [ ]:
run('profile')

## Fixed-exposure arms
Run each cell separately. Require `family_arm_complete` each time. Do not inspect evaluation or retune until all three finish.

In [ ]:
run('train','--arm','mask')

In [ ]:
run('train','--arm','core')

In [ ]:
run('train','--arm','full')

## Evaluate every requested row
The original tree is a reference, not the matched capacity control. Final training RMSE is descriptive. Three contrasts share an adjustment across the six planned comparisons in the two rounds. Reused games are not untouched confirmation.

In [ ]:
run('evaluate')
s=json.loads((OUT/'summary.json').read_text())
print(json.dumps({'metrics':s['metrics'],'contrasts':s['contrasts'],'core_ready':s['core_ready_for_replication'],'full_ready':s['full_ready_for_replication']},indent=2))
show(visuals.learning(OUT,ROUND),'learning')
show(visuals.metrics(OUT,ROUND),'matched_metrics')
show(visuals.intervals(OUT,ROUND),'paired_intervals')
show(visuals.horizons(OUT,ROUND),'horizon_errors')

## Fresh-process replay and report
Require `family_replay_exact` and `new_optimizer_steps: 0`. Replay will not manufacture missing models or predictions. No later fold launches automatically.

In [ ]:
run('replay')
run('report')
print(OUT/f'nfl_feature_round{ROUND}_report.zip')

## Decision
A feature needs at least 1% matched improvement and a negative adjusted upper game-bootstrap bound. The full arm must beat both core and mask. Another fold additionally requires no worse RMSE than the preserved tree. Proceed to the other independently frozen round unchanged after this round completes and replays—even if this feature fails. Stop both on integrity/runtime failures; a feature-specific budget stop must be reported, not bypassed. After both rounds, return both reports before more experiments.